# GNN-as-Judge + Qwen3-4B-Instruct-2507 LoRA rank 8 Colab 예제

이 notebook은 repo의 **GNN-as-Judge** 파이프라인을 Colab에서 작은 규모로 재현합니다.

1. few-shot graph split 생성 및 GNN judge 학습
2. Qwen/Qwen3-4B-Instruct-2507에 4-bit QLoRA LoRA rank 8 부착 후 초기 SFT
3. 논문 기법의 influential unlabeled node 선택
4. LLM/GNN 합의 및 GNN confidence 기반 pseudo-label 선별
5. GNN-as-Judge weak-supervised SFT, 선택적으로 DPO-style preference update
6. held-out test split 예측 및 accuracy / macro-F1 평가

Qwen3-4B-Instruct-2507 model card는 `transformers>=4.51.0` 사용을 권장하며, 이 notebook도 vendored `LLaMA-Factory` 대신 최신 `transformers` + `peft` 학습 루프를 사용합니다. 모델 카드: https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507


## 0. Runtime

Colab 메뉴에서 `Runtime -> Change runtime type -> GPU`를 선택하세요. 무료 T4에서는 `MAX_SELECTED_NODES`, `MAX_TEST_SAMPLES`, epoch 값을 작게 유지하는 것이 좋습니다.


In [ ]:
#@title 1. Repo 준비
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tally0818/CS471.git"  #@param {type:"string"}
REPO_DIR = "/content/GNN-as-Judge"  #@param {type:"string"}

repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

candidates = [repo_dir]
candidates += [p.parent.parent for p in repo_dir.glob("**/GNN/main.py")]
project_dir = next((p for p in candidates if (p / "GNN" / "main.py").exists()), None)
if project_dir is None:
    raise RuntimeError(f"GNN-as-Judge project root not found under {repo_dir}")

os.chdir(project_dir)
print("PROJECT_DIR =", Path.cwd())


In [ ]:
#@title 2. Dependencies 설치
import subprocess
import sys
import torch

def pip_install(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))

pip_install([
    "-U",
    "transformers>=4.51.0",
    "accelerate>=0.34.0",
    "peft>=0.13.0",
    "bitsandbytes>=0.43.1",
    "datasets>=2.16.0",
    "gdown",
    "ogb",
    "networkx",
    "scikit-learn",
    "tqdm",
])

torch_ver = torch.__version__.split("+")[0]
cuda_tag = "cpu"
if torch.cuda.is_available() and torch.version.cuda:
    cuda_tag = "cu" + torch.version.cuda.replace(".", "")
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"

try:
    pip_install(["torch-geometric", "pyg-lib", "torch-scatter", "torch-sparse", "-f", wheel_url])
except subprocess.CalledProcessError:
    print("PyG CUDA wheel install failed; falling back to torch-geometric only.")
    pip_install(["torch-geometric"])

print("torch:", torch.__version__, "cuda:", torch.version.cuda, "gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


In [ ]:
#@title 3. Experiment 설정
from pathlib import Path
import torch

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  #@param {type:"string"}
DATASET = "cora"  #@param ["cora", "citeseer", "pubmed", "arxiv"]
SEED = 42  #@param {type:"integer"}

# === GNN-as-Judge 모드 설정 ===
JUDGE_MODE = "Hetero-Ensemble"  #@param ["Single", "Homo-Ensemble", "Hetero-Ensemble"]
HOMO_ENSEMBLE_K = 3  #@param [3, 5] {type:"integer"}

# === 고정 설정 ===
SHOTS_LIST = [3, 5, 10]
VOTING_METHOD = "soft"
LORA_RANK = 8
LORA_ALPHA = 16
MAX_SELECTED_NODES = 60
MAX_TEST_SAMPLES = 100
INITIAL_SFT_EPOCHS = 3
WSFT_EPOCHS = 1
RUN_DPO_STAGE = True
DPO_EPOCHS = 1
CONFIDENCE_THRESHOLD = 0.7
CUTOFF_LEN = 1536

# === 경로 (multi-shot runner가 shot별로 재설정하므로 기본값만 세팅) ===
SHOTS = SHOTS_LIST[0]
BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "datasets"
RUN_ID = f"{DATASET}_{SHOTS}shot_seed{SEED}"
WORK_DIR = BASE_DIR / "results" / "colab_qwen3_gnn_as_judge" / RUN_ID
DATA_DIR = WORK_DIR / "data"
SFT_ADAPTER_DIR = BASE_DIR / "results" / "shared_initial_sft" / RUN_ID
WSFT_ADAPTER_DIR = WORK_DIR / "qwen3_lora8_gnn_as_judge_wsft"
DPO_ADAPTER_DIR = WORK_DIR / "qwen3_lora8_gnn_as_judge_dpo"

for p in [DATASET_DIR, WORK_DIR, DATA_DIR, SFT_ADAPTER_DIR, WSFT_ADAPTER_DIR, DPO_ADAPTER_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"DATASET = {DATASET}")
print(f"JUDGE_MODE = {JUDGE_MODE}")
print(f"SHOTS_LIST = {SHOTS_LIST}")
print(f"DEVICE = {DEVICE}")
if JUDGE_MODE == "Homo-Ensemble":
    print(f"  K = {HOMO_ENSEMBLE_K}")
elif JUDGE_MODE == "Hetero-Ensemble":
    print(f"  models = [GCN, GAT, SAGE]")

In [ ]:
#@title 4. Graph dataset 다운로드
import gdown
import json
import shutil
import tarfile
import zipfile
from pathlib import Path

dataset_pt = DATASET_DIR / f"{DATASET}.pt"
if not dataset_pt.exists():
    url = "https://drive.google.com/uc?id=14GmRVwhP1pUD_OIhoJU3oATZWTnklhPG"
    download_path = Path("/content/gnn_as_judge_datasets_download")
    unpack_dir = Path("/content/gnn_as_judge_datasets_unpacked")
    unpack_dir.mkdir(parents=True, exist_ok=True)
    gdown.download(url=url, output=str(download_path), quiet=False)

    if zipfile.is_zipfile(download_path):
        with zipfile.ZipFile(download_path) as zf:
            zf.extractall(unpack_dir)
    elif tarfile.is_tarfile(download_path):
        with tarfile.open(download_path) as tf:
            tf.extractall(unpack_dir)
    elif download_path.suffix == ".pt":
        shutil.copy2(download_path, DATASET_DIR / download_path.name)
    else:
        print("Downloaded file is not an archive; searching it and unpack dir for .pt files.")

    for pt in list(unpack_dir.rglob("*.pt")) + list(Path("/content").glob("*.pt")):
        shutil.copy2(pt, DATASET_DIR / pt.name)

# === arxiv fallback: Drive에 없으면 OGB에서 자동 생성 ===
if DATASET == "arxiv" and not dataset_pt.exists():
    print("arxiv.pt not found in archive. Building from OGB...")
    import torch
    import numpy as np
    import pandas as pd
    import gzip
    from ogb.nodeproppred import PygNodePropPredDataset
    from torch_geometric.data import Data
    from torch_geometric.utils import to_undirected

    ogb_dataset = PygNodePropPredDataset(name="ogbn-arxiv", root="/content/ogb_data")
    ogb_data = ogb_dataset[0]
    split_idx = ogb_dataset.get_idx_split()

    num_nodes = ogb_data.x.shape[0]
    y = ogb_data.y.squeeze(-1)

    # edge_index
    if hasattr(ogb_data, "adj_t"):
        row, col, _ = ogb_data.adj_t.t().coo()
        edge_index = torch.stack([row, col], dim=0)
    else:
        edge_index = ogb_data.edge_index
    edge_index = to_undirected(edge_index)

    # masks
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    train_mask[split_idx["train"]] = True
    val_mask[split_idx["valid"]] = True
    test_mask[split_idx["test"]] = True

    # label_name: OGB 공식 매핑 파일에서 읽어서 인덱스 순서 보장
    mapping_dir = Path("/content/ogb_data/ogbn_arxiv/mapping")
    label_map_file = mapping_dir / "labelidx2arxivcatid.csv.gz"
    if label_map_file.exists():
        label_df = pd.read_csv(label_map_file)
        label_df = label_df.sort_values(label_df.columns[0])
        label_name = []
        for _, row in label_df.iterrows():
            cat_id = str(row.iloc[1]).strip()
            label_name.append(f"arxiv {cat_id.replace('.', ' ').lower()}")
        print(f"Loaded {len(label_name)} label names from OGB mapping")
    else:
        print("Warning: labelidx2arxivcatid.csv.gz not found, using hardcoded names")
        label_name = [
            'arxiv cs ai', 'arxiv cs ar', 'arxiv cs cc', 'arxiv cs ce', 'arxiv cs cg',
            'arxiv cs cl', 'arxiv cs cr', 'arxiv cs cv', 'arxiv cs cy', 'arxiv cs db',
            'arxiv cs dc', 'arxiv cs dl', 'arxiv cs dm', 'arxiv cs ds', 'arxiv cs et',
            'arxiv cs fl', 'arxiv cs gl', 'arxiv cs gr', 'arxiv cs gt', 'arxiv cs hc',
            'arxiv cs ir', 'arxiv cs it', 'arxiv cs lg', 'arxiv cs lo', 'arxiv cs ma',
            'arxiv cs mm', 'arxiv cs ms', 'arxiv cs na', 'arxiv cs ne', 'arxiv cs ni',
            'arxiv cs oh', 'arxiv cs os', 'arxiv cs pf', 'arxiv cs pl', 'arxiv cs ro',
            'arxiv cs sc', 'arxiv cs sd', 'arxiv cs se', 'arxiv cs si', 'arxiv cs sy',
        ]

    # raw_texts: 제목+초록
    titleabs_file = mapping_dir / "titleabs.tsv.gz"
    if titleabs_file.exists():
        raw_texts = []
        with gzip.open(titleabs_file, "rt", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t")
                title = parts[1] if len(parts) > 1 else ""
                abstract = parts[2] if len(parts) > 2 else ""
                raw_texts.append(f"Title: {title}\nAbstract: {abstract}" if abstract else f"Title: {title}")
        print(f"Loaded {len(raw_texts)} title+abstract texts")
    else:
        print("Warning: titleabs.tsv.gz not found, using placeholder texts")
        raw_texts = [f"arxiv paper node {i}" for i in range(num_nodes)]

    graph_data = Data(
        x=ogb_data.x,
        edge_index=edge_index,
        y=y,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask,
    )
    graph_data.raw_texts = raw_texts
    graph_data.label_name = label_name

    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(graph_data, dataset_pt)
    print(f"Saved {dataset_pt} ({num_nodes} nodes, {edge_index.shape[1]} edges, {len(label_name)} classes)")

if not dataset_pt.exists():
    raise FileNotFoundError(f"{dataset_pt} not found. Put {DATASET}.pt under {DATASET_DIR} and rerun this cell.")

print("Available graph datasets:", sorted(p.name for p in DATASET_DIR.glob("*.pt")))

In [ ]:
#@title 5. GNN judge 학습
import subprocess
import sys

gnn_base_cmd = [
    sys.executable,
    "main.py",
    "--dataset", DATASET,
    "--shots", str(SHOTS),
    "--hidden_dim", "64",
    "--n_layers", "2",
    "--epochs", "200",
    "--patience", "50",
    "--seed", str(SEED),
    "--run_times", "1",
    "--print_freq", "50",
    "--device", DEVICE,
]

if JUDGE_MODE == "Hetero-Ensemble":
    # Train GCN, GAT, SAGE separately via --hetero_ensemble flag
    hetero_cmd = gnn_base_cmd + ["--gnn_type", "GCN", "--hetero_ensemble"]
    print("Running hetero-ensemble training (GCN, GAT, SAGE)...")
    subprocess.run(hetero_cmd, cwd=str(BASE_DIR / "GNN"), check=True)

    HETERO_TYPES = ["GCN", "GAT", "SAGE"]
    GNN_HETERO_MODEL_PATHS = [
        BASE_DIR / "results" / "GNN" / f"{DATASET}_{SHOTS}_shot_best_model_{t}.pt"
        for t in HETERO_TYPES
    ]
    for p in GNN_HETERO_MODEL_PATHS:
        if not p.exists():
            raise FileNotFoundError(p)
    print(f"Hetero-Ensemble models: {[p.name for p in GNN_HETERO_MODEL_PATHS]}")

elif JUDGE_MODE == "Homo-Ensemble":
    gnn_cmd = gnn_base_cmd + [
        "--gnn_type", "GCN",
        "--ensemble_k", str(HOMO_ENSEMBLE_K),
    ]
    subprocess.run(gnn_cmd, cwd=str(BASE_DIR / "GNN"), check=True)

    GNN_MODEL_PATHS = [
        BASE_DIR / "results" / "GNN" / f"{DATASET}_{SHOTS}_shot_best_model_ensemble{i}.pt"
        for i in range(HOMO_ENSEMBLE_K)
    ]
    for p in GNN_MODEL_PATHS:
        if not p.exists():
            raise FileNotFoundError(p)
    print(f"Homo-Ensemble GNN models ({HOMO_ENSEMBLE_K}): {[p.name for p in GNN_MODEL_PATHS]}")

else:  # Single
    gnn_cmd = gnn_base_cmd + ["--gnn_type", "GCN"]
    subprocess.run(gnn_cmd, cwd=str(BASE_DIR / "GNN"), check=True)

    GNN_MODEL_PATH = BASE_DIR / "results" / "GNN" / f"{DATASET}_{SHOTS}_shot_best_model_run0.pt"
    if not GNN_MODEL_PATH.exists():
        raise FileNotFoundError(GNN_MODEL_PATH)
    print("GNN_MODEL_PATH =", GNN_MODEL_PATH)

# --- Report GNN parameter counts ---
import torch
from common import GNNEncoder, create_few_shot_dataset

_gdata = create_few_shot_dataset(DATASET, shots=SHOTS, seed=SEED, device=DEVICE, path_prefix=".")
_nclass = _gdata.y.max().item() + 1
_indim = _gdata.x.shape[1]

if JUDGE_MODE == "Hetero-Ensemble":
    total_params = 0
    for gtype in HETERO_TYPES:
        m = GNNEncoder(_indim, 64, _nclass, n_layers=2, gnn_type=gtype)
        p = sum(p.numel() for p in m.parameters())
        total_params += p
        print(f"  {gtype}: {p:,} params")
    print(f"  Hetero-Ensemble total: {total_params:,} params")
elif JUDGE_MODE == "Homo-Ensemble":
    m = GNNEncoder(_indim, 64, _nclass, n_layers=2, gnn_type="GCN")
    p = sum(p.numel() for p in m.parameters())
    print(f"  GCN x{HOMO_ENSEMBLE_K}: {p:,} params each, total {p * HOMO_ENSEMBLE_K:,} params")
else:
    m = GNNEncoder(_indim, 64, _nclass, n_layers=2, gnn_type="GCN")
    p = sum(p.numel() for p in m.parameters())
    print(f"  Single GCN: {p:,} params")

del _gdata, m

In [ ]:
#@title 6. Few-shot SFT / validation / test / unlabeled JSON 생성
import subprocess
import sys

sft_output_base = DATA_DIR / f"{DATASET}_sft.json"
create_sft_cmd = [
    sys.executable,
    "create_sft.py",
    "--dataset", DATASET,
    "--output", str(sft_output_base),
    "--shots", str(SHOTS),
    "--seed", str(SEED),
    "--max_test_samples", str(MAX_TEST_SAMPLES),
    "--device", DEVICE,
    "--path_prefix", ".",
]
subprocess.run(create_sft_cmd, cwd=str(BASE_DIR), check=True)

SFT_PREFIX = f"{DATASET}_sft_{SHOTS}_shot"
SFT_TRAIN_JSON = DATA_DIR / f"{SFT_PREFIX}_train.json"
SFT_VAL_JSON = DATA_DIR / f"{SFT_PREFIX}_val.json"
SFT_TEST_JSON = DATA_DIR / f"{SFT_PREFIX}_test.json"
UNLABELED_JSON = DATA_DIR / f"{SFT_PREFIX}_unlabeled.json"
UNLABELED_NODE_IDS_JSON = DATA_DIR / f"{DATASET}_{SHOTS}_shot_unlabeled_node_ids.json"

for p in [SFT_TRAIN_JSON, SFT_TEST_JSON, UNLABELED_JSON, UNLABELED_NODE_IDS_JSON]:
    if not p.exists():
        raise FileNotFoundError(p)
    print(p.name, "exists")


In [ ]:
# Training / inference utility functions
import gc
import json
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def assistant_text(row):
    for turn in row.get("conversations", []):
        if turn.get("from") == "gpt":
            return turn.get("value", "")
    return ""

def user_text(row):
    for turn in row.get("conversations", []):
        if turn.get("from") == "human":
            return turn.get("value", "")
    return ""

def tokenize_chat_example(tokenizer, prompt, answer, max_length):
    user_messages = [{"role": "user", "content": prompt}]
    full_messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]
    prompt_text = tokenizer.apply_chat_template(user_messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=max_length).input_ids
    labels = full_ids.copy()
    mask_until = min(len(prompt_ids), len(labels))
    labels[:mask_until] = [-100] * mask_until
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

class ShareGPTSFTDataset(Dataset):
    def __init__(self, json_path, tokenizer, max_length, max_samples=None, shuffle=True):
        rows = json.load(open(json_path, encoding="utf-8"))
        rows = [r for r in rows if user_text(r) and assistant_text(r)]
        if shuffle:
            rng = random.Random(SEED)
            rng.shuffle(rows)
        if max_samples:
            rows = rows[:max_samples]
        self.items = [tokenize_chat_example(tokenizer, user_text(r), assistant_text(r), max_length) for r in rows]
        print(f"Loaded {len(self.items)} SFT examples from {json_path}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

class CausalCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        pad_id = self.tokenizer.pad_token_id
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [pad_id] * pad_len)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)
            batch["labels"].append(f["labels"] + [-100] * pad_len)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

def load_qwen3_lora_model():
    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=compute_dtype,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer

def train_sft(model, tokenizer, json_path, output_dir, epochs, lr, grad_accum=8, max_samples=None):
    dataset = ShareGPTSFTDataset(json_path, tokenizer, CUTOFF_LEN, max_samples=max_samples)
    if len(dataset) == 0:
        print(f"No examples in {json_path}; skipping SFT.")
        return
    args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        remove_unused_columns=False,
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=CausalCollator(tokenizer))
    trainer.train()
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def clean_prediction(text):
    text = text.strip()
    for marker in ["<|im_end|>", "</s>", "<|endoftext|>"]:
        text = text.split(marker)[0].strip()
    return text.split("\n")[0].strip()

@torch.no_grad()
def generate_predictions(model, tokenizer, input_json, output_jsonl, max_samples=None, max_new_tokens=16):
    rows = json.load(open(input_json, encoding="utf-8"))
    if max_samples:
        rows = rows[:max_samples]
    model.eval()
    model.config.use_cache = True
    device = next(model.parameters()).device
    output_jsonl = Path(output_jsonl)
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)
    with open(output_jsonl, "w", encoding="utf-8") as f:
        for row in tqdm(rows, desc=f"Generating {Path(input_json).name}"):
            prompt = user_text(row)
            label = assistant_text(row)
            chat = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
            inputs = tokenizer(chat, return_tensors="pt", truncation=True, max_length=CUTOFF_LEN).to(device)
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            new_tokens = generated[0, inputs["input_ids"].shape[-1]:]
            pred = clean_prediction(tokenizer.decode(new_tokens, skip_special_tokens=True))
            f.write(json.dumps({"predict": pred, "label": label}, ensure_ascii=False) + "\n")
    model.config.use_cache = False
    print(f"Saved predictions to {output_jsonl}")


In [ ]:
#@title 7. Qwen3-4B-Instruct-2507 + LoRA rank 8 초기 SFT
from peft import PeftModel

seed_everything(SEED)

sft_done_marker = SFT_ADAPTER_DIR / "adapter_config.json"
if sft_done_marker.exists():
    print(f"Initial SFT adapter already exists at {SFT_ADAPTER_DIR}. Loading...")
    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=compute_dtype,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR))
    model.config.use_cache = False
    print("Loaded existing initial SFT adapter (shared across judge modes).")
else:
    print(f"No existing SFT adapter found. Training new one...")
    model, tokenizer = load_qwen3_lora_model()
    train_sft(
        model=model,
        tokenizer=tokenizer,
        json_path=SFT_TRAIN_JSON,
        output_dir=SFT_ADAPTER_DIR,
        epochs=INITIAL_SFT_EPOCHS,
        lr=2e-4,
        grad_accum=8,
    )
    print(f"Initial SFT adapter saved to {SFT_ADAPTER_DIR}")
    print("Subsequent runs with different JUDGE_MODE will reuse this adapter.")

In [ ]:
#@title 8. Influential unlabeled node 선택 및 selected dataset 생성
import json
import subprocess
import sys

SELECTED_NODES_JSON = WORK_DIR / f"{RUN_ID}_selected_nodes.json"
select_cmd = [
    sys.executable,
    "select_influential_nodes.py",
    "--dataset", DATASET,
    "--k", str(MAX_SELECTED_NODES),
    "--output_file", str(SELECTED_NODES_JSON),
    "--shots", str(SHOTS),
    "--seed", str(SEED),
    "--method", "auto",
    "--max_subgraph_nodes", "1500",
    "--max_distance", "3",
    "--path_prefix", ".",
]
subprocess.run(select_cmd, cwd=str(BASE_DIR), check=True)

with open(SELECTED_NODES_JSON) as f:
    selected_ids = set(json.load(f)["selected_node_ids"])
with open(UNLABELED_NODE_IDS_JSON) as f:
    all_unlabeled_ids = json.load(f)["selected_node_ids"]
with open(UNLABELED_JSON, encoding="utf-8") as f:
    unlabeled_rows = json.load(f)

filtered_rows = []
ordered_ids = []
for node_id, row in zip(all_unlabeled_ids, unlabeled_rows):
    if int(node_id) in selected_ids:
        filtered_rows.append(row)
        ordered_ids.append(int(node_id))

SELECTED_DATASET_JSON = DATA_DIR / f"{SFT_PREFIX}_selected.json"
ORDERED_SELECTED_NODES_JSON = WORK_DIR / f"{RUN_ID}_selected_nodes_ordered.json"
with open(SELECTED_DATASET_JSON, "w", encoding="utf-8") as f:
    json.dump(filtered_rows, f, ensure_ascii=False, indent=2)
with open(ORDERED_SELECTED_NODES_JSON, "w", encoding="utf-8") as f:
    json.dump({"selected_node_ids": ordered_ids}, f, indent=2)

print(f"Selected dataset: {len(filtered_rows)} samples")
print("SELECTED_DATASET_JSON =", SELECTED_DATASET_JSON)
print("ORDERED_SELECTED_NODES_JSON =", ORDERED_SELECTED_NODES_JSON)


In [ ]:
#@title 9. 선택된 node에 대한 Qwen prediction 생성
SELECTED_LLM_PRED_JSONL = WORK_DIR / f"{RUN_ID}_selected_llm_preds.jsonl"
generate_predictions(
    model=model,
    tokenizer=tokenizer,
    input_json=SELECTED_DATASET_JSON,
    output_jsonl=SELECTED_LLM_PRED_JSONL,
    max_new_tokens=16,
)


In [ ]:
#@title 10. GNN-as-Judge WSFT / DPO 데이터 생성
import json
import subprocess
import sys

DPO_JSON = DATA_DIR / f"{RUN_ID}_gnn_as_judge_dpo.json"
WSFT_JSON = DATA_DIR / f"{RUN_ID}_gnn_as_judge_wsft.json"

# Base command (shared across all modes)
create_wsft_cmd = [
    sys.executable,
    "create_wsft.py",
    "--dataset", DATASET,
    "--selected_nodes_path", str(ORDERED_SELECTED_NODES_JSON),
    "--llm_predictions", str(SELECTED_LLM_PRED_JSONL),
    "--dpo_output_path", str(DPO_JSON),
    "--sft_output_path", str(WSFT_JSON),
    "--confidence_threshold", str(CONFIDENCE_THRESHOLD),
    "--shots", str(SHOTS),
    "--gnn_type", "GCN",
    "--hidden_dim", "64",
    "--n_layers", "2",
    "--seed", str(SEED),
    "--device", DEVICE,
    "--path_prefix", ".",
]

if JUDGE_MODE == "Hetero-Ensemble":
    # pretrained_model is unused but required arg — point to GCN checkpoint
    create_wsft_cmd += [
        "--pretrained_model", str(BASE_DIR / "results" / "GNN" / f"{DATASET}_{SHOTS}_shot_best_model_GCN.pt"),
        "--hetero_models", "GCN", "GAT", "SAGE",
        "--ensemble_model_dir", str(BASE_DIR / "results" / "GNN"),
        "--voting_method", VOTING_METHOD,
    ]
elif JUDGE_MODE == "Homo-Ensemble":
    create_wsft_cmd += [
        "--pretrained_model", str(BASE_DIR / "results" / "GNN" / f"{DATASET}_{SHOTS}_shot_best_model_ensemble0.pt"),
        "--use_ensemble",
        "--ensemble_k", str(HOMO_ENSEMBLE_K),
        "--ensemble_model_dir", str(BASE_DIR / "results" / "GNN"),
        "--voting_method", VOTING_METHOD,
    ]
else:  # Single
    create_wsft_cmd += [
        "--pretrained_model", str(GNN_MODEL_PATH),
    ]

subprocess.run(create_wsft_cmd, cwd=str(BASE_DIR), check=True)

wsft_rows = json.load(open(WSFT_JSON, encoding="utf-8"))
dpo_rows = json.load(open(DPO_JSON, encoding="utf-8"))
num_real_pref = sum(1 for r in dpo_rows if r.get("chosen", {}).get("value") != r.get("rejected", {}).get("value"))
print(f"WSFT examples: {len(wsft_rows)}")
print(f"DPO rows: {len(dpo_rows)} | non-trivial preference pairs: {num_real_pref}")
print("WSFT_JSON =", WSFT_JSON)
print("DPO_JSON =", DPO_JSON)

In [ ]:
#@title 11. GNN-as-Judge weak-supervised SFT
train_sft(
    model=model,
    tokenizer=tokenizer,
    json_path=WSFT_JSON,
    output_dir=WSFT_ADAPTER_DIR,
    epochs=WSFT_EPOCHS,
    lr=5e-5,
    grad_accum=4,
)


In [ ]:
# Optional DPO-style preference update utilities
class PreferenceDataset(Dataset):
    def __init__(self, json_path, tokenizer, max_length):
        rows = json.load(open(json_path, encoding="utf-8"))
        self.items = []
        for row in rows:
            prompt = user_text(row)
            chosen = row.get("chosen", {}).get("value", "")
            rejected = row.get("rejected", {}).get("value", "")
            if not prompt or not chosen or not rejected or chosen == rejected:
                continue
            self.items.append({
                "chosen": tokenize_chat_example(tokenizer, prompt, chosen, max_length),
                "rejected": tokenize_chat_example(tokenizer, prompt, rejected, max_length),
            })
        print(f"Loaded {len(self.items)} non-trivial preference pairs from {json_path}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

class PreferenceCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.sft_collator = CausalCollator(tokenizer)

    def __call__(self, features):
        return {
            "chosen": self.sft_collator([f["chosen"] for f in features]),
            "rejected": self.sft_collator([f["rejected"] for f in features]),
        }

def move_batch(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

def sequence_logps(model, batch):
    outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits = outputs.logits[:, :-1, :]
    target = batch["input_ids"][:, 1:]
    loss_mask = batch["labels"][:, 1:] != -100
    log_probs = torch.log_softmax(logits, dim=-1)
    token_logps = torch.gather(log_probs, dim=-1, index=target.unsqueeze(-1)).squeeze(-1)
    return (token_logps * loss_mask).sum(dim=-1)

def run_dpo_update(model, tokenizer, dpo_json, output_dir, epochs=1, beta=0.1, lr=1e-6, grad_accum=2):
    dataset = PreferenceDataset(dpo_json, tokenizer, CUTOFF_LEN)
    if len(dataset) == 0:
        print("No non-trivial preference pairs; skipping DPO stage.")
        return
    loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=PreferenceCollator(tokenizer))
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    device = next(model.parameters()).device
    model.train()
    model.config.use_cache = False
    global_step = 0
    optimizer.zero_grad(set_to_none=True)
    for epoch in range(epochs):
        pbar = tqdm(loader, desc=f"DPO epoch {epoch + 1}/{epochs}")
        for step, batch in enumerate(pbar, start=1):
            chosen = move_batch(batch["chosen"], device)
            rejected = move_batch(batch["rejected"], device)
            with torch.no_grad():
                with model.disable_adapter():
                    ref_chosen = sequence_logps(model, chosen)
                    ref_rejected = sequence_logps(model, rejected)
            pi_chosen = sequence_logps(model, chosen)
            pi_rejected = sequence_logps(model, rejected)
            pi_logratio = pi_chosen - pi_rejected
            ref_logratio = ref_chosen - ref_rejected
            loss = -F.logsigmoid(beta * (pi_logratio - ref_logratio)).mean()
            (loss / grad_accum).backward()
            if step % grad_accum == 0 or step == len(loader):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
            pbar.set_postfix(loss=float(loss.detach().cpu()))
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
#@title 12. 선택 사항: DPO-style preference update
if RUN_DPO_STAGE:
    run_dpo_update(
        model=model,
        tokenizer=tokenizer,
        dpo_json=DPO_JSON,
        output_dir=DPO_ADAPTER_DIR,
        epochs=DPO_EPOCHS,
        beta=0.1,
        lr=1e-6,
        grad_accum=2,
    )
else:
    print("RUN_DPO_STAGE=False; DPO-style update skipped.")


In [ ]:
#@title 13. Final evaluation
import json
import subprocess
import sys

FINAL_TEST_PRED_JSONL = WORK_DIR / "qwen3_lora8_gnn_as_judge_test_predictions.jsonl"
generate_predictions(
    model=model,
    tokenizer=tokenizer,
    input_json=SFT_TEST_JSON,
    output_jsonl=FINAL_TEST_PRED_JSONL,
    max_samples=MAX_TEST_SAMPLES,
    max_new_tokens=16,
)

EVAL_DIR = WORK_DIR / "evaluation"
eval_cmd = [
    sys.executable,
    "evaluate_predictions.py",
    "--dataset", DATASET,
    "--pred_file", str(FINAL_TEST_PRED_JSONL),
    "--output_dir", str(EVAL_DIR),
    "--model_name", "qwen3_4b_lora8_gnn_as_judge",
    "--device", DEVICE,
    "--path_prefix", ".",
]
subprocess.run(eval_cmd, cwd=str(BASE_DIR), check=True)

metrics_path = EVAL_DIR / "qwen3_4b_lora8_gnn_as_judge" / "metrics.json"
metrics = json.load(open(metrics_path))
print(json.dumps(metrics, indent=2))
print("metrics_path =", metrics_path)


## 14. Confidence Threshold Sweep

GNN-as-Judge의 confidence threshold를 sweep하여 최적값을 탐색합니다.
앙상블 soft voting은 확률 분포를 평탄화하므로, Single GNN과 동일한 threshold가 최적이 아닐 수 있습니다.

셀 5~9 (GNN 학습, SFT, node selection, LLM prediction)의 결과를 재사용하고,
threshold만 변경하여 WSFT/DPO 데이터 생성 → 학습 → 평가를 반복합니다.

In [ ]:
#@title 14-1. Threshold Sweep 실행
import gc
import json
import subprocess
import sys
import numpy as np
from pathlib import Path
from peft import PeftModel

THRESHOLDS = [0.3, 0.5, 0.7, 0.9]  #@param {type:"raw"}

# 셀 9까지의 결과물 경로 재사용
assert ORDERED_SELECTED_NODES_JSON.exists(), "셀 8을 먼저 실행하세요"
assert SELECTED_LLM_PRED_JSONL.exists(), "셀 9를 먼저 실행하세요"

sweep_results = []

for thresh in THRESHOLDS:
    print(f"\n{'='*60}")
    print(f"  Threshold = {thresh}")
    print(f"{'='*60}\n")

    sweep_dir = WORK_DIR / f"sweep_thresh_{thresh}"
    sweep_data = sweep_dir / "data"
    sweep_wsft_adapter = sweep_dir / "wsft_adapter"
    sweep_dpo_adapter = sweep_dir / "dpo_adapter"
    for p in [sweep_data, sweep_wsft_adapter, sweep_dpo_adapter]:
        p.mkdir(parents=True, exist_ok=True)

    # --- 1. create_wsft with this threshold ---
    dpo_json = sweep_data / f"{RUN_ID}_dpo.json"
    wsft_json = sweep_data / f"{RUN_ID}_wsft.json"

    create_wsft_cmd = [
        sys.executable, "create_wsft.py",
        "--dataset", DATASET,
        "--selected_nodes_path", str(ORDERED_SELECTED_NODES_JSON),
        "--llm_predictions", str(SELECTED_LLM_PRED_JSONL),
        "--dpo_output_path", str(dpo_json),
        "--sft_output_path", str(wsft_json),
        "--confidence_threshold", str(thresh),
        "--shots", str(SHOTS),
        "--gnn_type", "GCN",
        "--hidden_dim", "64",
        "--n_layers", "2",
        "--seed", str(SEED),
        "--device", DEVICE,
        "--path_prefix", ".",
    ]

    if JUDGE_MODE == "Hetero-Ensemble":
        create_wsft_cmd += [
            "--hetero_models", "GCN", "GAT", "SAGE",
            "--ensemble_model_dir", str(BASE_DIR / "results" / "GNN"),
            "--voting_method", VOTING_METHOD,
        ]
    elif JUDGE_MODE == "Homo-Ensemble":
        create_wsft_cmd += [
            "--pretrained_model", str(BASE_DIR / "results" / "GNN" / f"{DATASET}_{SHOTS}_shot_best_model_ensemble0.pt"),
            "--use_ensemble", "--ensemble_k", str(HOMO_ENSEMBLE_K),
            "--ensemble_model_dir", str(BASE_DIR / "results" / "GNN"),
            "--voting_method", VOTING_METHOD,
        ]
    else:
        create_wsft_cmd += [
            "--pretrained_model", str(GNN_MODEL_PATH),
        ]

    subprocess.run(create_wsft_cmd, cwd=str(BASE_DIR), check=True)

    wsft_rows = json.load(open(wsft_json, encoding="utf-8"))
    dpo_rows = json.load(open(dpo_json, encoding="utf-8"))
    num_real_pref = sum(1 for r in dpo_rows if r.get("chosen", {}).get("value") != r.get("rejected", {}).get("value"))
    print(f"  WSFT: {len(wsft_rows)} examples, DPO: {num_real_pref} non-trivial pairs")

    # --- 2. Load fresh model from initial SFT adapter ---
    seed_everything(SEED)
    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, torch_dtype=compute_dtype,
        quantization_config=bnb_config, device_map="auto",
    )
    sweep_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR))
    sweep_model.config.use_cache = False

    # --- 3. WSFT ---
    train_sft(
        model=sweep_model, tokenizer=tokenizer, json_path=wsft_json,
        output_dir=sweep_wsft_adapter, epochs=WSFT_EPOCHS, lr=5e-5, grad_accum=4,
    )

    # --- 4. DPO (optional) ---
    if RUN_DPO_STAGE and num_real_pref > 0:
        run_dpo_update(
            model=sweep_model, tokenizer=tokenizer, dpo_json=dpo_json,
            output_dir=sweep_dpo_adapter, epochs=DPO_EPOCHS, beta=0.1, lr=1e-6, grad_accum=2,
        )

    # --- 5. Evaluate ---
    sweep_pred = sweep_dir / "test_predictions.jsonl"
    generate_predictions(
        model=sweep_model, tokenizer=tokenizer, input_json=SFT_TEST_JSON,
        output_jsonl=sweep_pred, max_samples=MAX_TEST_SAMPLES, max_new_tokens=16,
    )

    eval_dir = sweep_dir / "evaluation"
    subprocess.run([
        sys.executable, "evaluate_predictions.py",
        "--dataset", DATASET, "--pred_file", str(sweep_pred),
        "--output_dir", str(eval_dir),
        "--model_name", "sweep", "--device", DEVICE, "--path_prefix", ".",
    ], cwd=str(BASE_DIR), check=True)

    metrics = json.load(open(eval_dir / "sweep" / "metrics.json"))
    metrics["threshold"] = thresh
    metrics["wsft_count"] = len(wsft_rows)
    metrics["dpo_pref_count"] = num_real_pref
    sweep_results.append(metrics)
    print(f"  → Accuracy: {metrics.get('accuracy', 'N/A'):.4f}, Macro-F1: {metrics.get('macro_f1', 'N/A'):.4f}")

    # Cleanup
    del sweep_model, base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Summary ---
print(f"\n{'='*60}")
print(f"  Threshold Sweep Summary | {JUDGE_MODE} | {DATASET} {SHOTS}-shot")
print(f"{'='*60}")
print(f"{'Threshold':>10} {'Accuracy':>10} {'Macro-F1':>10} {'WSFT#':>8} {'DPO#':>8}")
print(f"{'-'*46}")
for r in sweep_results:
    print(f"{r['threshold']:>10.1f} {r.get('accuracy', 0):>10.4f} {r.get('macro_f1', 0):>10.4f} {r['wsft_count']:>8} {r['dpo_pref_count']:>8}")

best = max(sweep_results, key=lambda x: x.get("accuracy", 0))
print(f"\nBest threshold: {best['threshold']} (Accuracy={best.get('accuracy', 0):.4f})")

## 15. Multi-shot runner (3 / 5 / 10-shot 일괄 실행)

`SHOTS_LIST`의 모든 shot 수에 대해 전체 파이프라인(GNN 학습 → SFT → node selection → LLM prediction → WSFT → DPO → 평가)을 자동 실행하고, 결과를 비교표로 출력합니다.

셀 4(dataset 다운로드)까지 실행한 후 이 셀을 실행하세요. 셀 5~13의 수동 실행은 불필요합니다.

In [ ]:
#@title 15-1. Multi-shot 일괄 실행
import gc
import json
import subprocess
import sys
import numpy as np
from pathlib import Path
from peft import PeftModel


def run_pipeline_for_shots(shots):
    """Run full pipeline for a given shot count. Returns metrics dict."""
    print(f"\n{'='*60}")
    print(f"  {JUDGE_MODE} | {DATASET} | {shots}-shot | seed={SEED}")
    print(f"{'='*60}\n")

    run_id = f"{DATASET}_{shots}shot_seed{SEED}"
    work_dir = BASE_DIR / "results" / "colab_qwen3_gnn_as_judge" / run_id
    data_dir = work_dir / "data"
    sft_adapter_dir = BASE_DIR / "results" / "shared_initial_sft" / run_id
    wsft_adapter_dir = work_dir / "qwen3_lora8_gnn_as_judge_wsft"
    dpo_adapter_dir = work_dir / "qwen3_lora8_gnn_as_judge_dpo"
    for p in [work_dir, data_dir, sft_adapter_dir, wsft_adapter_dir, dpo_adapter_dir]:
        p.mkdir(parents=True, exist_ok=True)

    # ---- Step 1: GNN Training ----
    print(f"[{shots}-shot] Step 1/7: GNN training")
    gnn_base_cmd = [
        sys.executable, "main.py",
        "--dataset", DATASET, "--shots", str(shots),
        "--hidden_dim", "64", "--n_layers", "2",
        "--epochs", "200", "--patience", "50",
        "--seed", str(SEED), "--run_times", "1",
        "--print_freq", "50", "--device", DEVICE,
    ]
    if JUDGE_MODE == "Hetero-Ensemble":
        subprocess.run(gnn_base_cmd + ["--gnn_type", "GCN", "--hetero_ensemble"],
                       cwd=str(BASE_DIR / "GNN"), check=True)
    elif JUDGE_MODE == "Homo-Ensemble":
        subprocess.run(gnn_base_cmd + ["--gnn_type", "GCN", "--ensemble_k", str(HOMO_ENSEMBLE_K)],
                       cwd=str(BASE_DIR / "GNN"), check=True)
    else:
        subprocess.run(gnn_base_cmd + ["--gnn_type", "GCN"],
                       cwd=str(BASE_DIR / "GNN"), check=True)

    # ---- Step 2: SFT data generation ----
    print(f"[{shots}-shot] Step 2/7: SFT data generation")
    sft_prefix = f"{DATASET}_sft_{shots}_shot"
    subprocess.run([
        sys.executable, "create_sft.py",
        "--dataset", DATASET,
        "--output", str(data_dir / f"{DATASET}_sft.json"),
        "--shots", str(shots), "--seed", str(SEED),
        "--max_test_samples", str(MAX_TEST_SAMPLES),
        "--device", DEVICE, "--path_prefix", ".",
    ], cwd=str(BASE_DIR), check=True)

    sft_train_json = data_dir / f"{sft_prefix}_train.json"
    sft_test_json = data_dir / f"{sft_prefix}_test.json"
    unlabeled_json = data_dir / f"{sft_prefix}_unlabeled.json"
    unlabeled_ids_json = data_dir / f"{DATASET}_{shots}_shot_unlabeled_node_ids.json"

    # ---- Step 3: Initial SFT ----
    print(f"[{shots}-shot] Step 3/7: Initial SFT")
    seed_everything(SEED)
    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
    )

    sft_done = (sft_adapter_dir / "adapter_config.json").exists()
    if sft_done:
        print(f"  Loading existing SFT adapter from {sft_adapter_dir}")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, trust_remote_code=True, torch_dtype=compute_dtype,
            quantization_config=bnb_config, device_map="auto",
        )
        mdl = PeftModel.from_pretrained(base_model, str(sft_adapter_dir))
        mdl.config.use_cache = False
    else:
        print(f"  Training new SFT adapter")
        mdl, tokenizer = load_qwen3_lora_model()
        train_sft(model=mdl, tokenizer=tokenizer, json_path=sft_train_json,
                  output_dir=sft_adapter_dir, epochs=INITIAL_SFT_EPOCHS, lr=2e-4, grad_accum=8)

    # ---- Step 4: Node selection ----
    print(f"[{shots}-shot] Step 4/7: Node selection")
    selected_json = work_dir / f"{run_id}_selected_nodes.json"
    subprocess.run([
        sys.executable, "select_influential_nodes.py",
        "--dataset", DATASET, "--k", str(MAX_SELECTED_NODES),
        "--output_file", str(selected_json),
        "--shots", str(shots), "--seed", str(SEED),
        "--method", "auto", "--max_subgraph_nodes", "1500",
        "--max_distance", "3", "--path_prefix", ".",
    ], cwd=str(BASE_DIR), check=True)

    with open(selected_json) as f:
        selected_ids = set(json.load(f)["selected_node_ids"])
    with open(unlabeled_ids_json) as f:
        all_unlabeled_ids = json.load(f)["selected_node_ids"]
    with open(unlabeled_json, encoding="utf-8") as f:
        unlabeled_rows = json.load(f)

    filtered_rows, ordered_ids = [], []
    for nid, row in zip(all_unlabeled_ids, unlabeled_rows):
        if int(nid) in selected_ids:
            filtered_rows.append(row)
            ordered_ids.append(int(nid))

    selected_dataset_json = data_dir / f"{sft_prefix}_selected.json"
    ordered_json = work_dir / f"{run_id}_selected_nodes_ordered.json"
    with open(selected_dataset_json, "w", encoding="utf-8") as f:
        json.dump(filtered_rows, f, ensure_ascii=False, indent=2)
    with open(ordered_json, "w", encoding="utf-8") as f:
        json.dump({"selected_node_ids": ordered_ids}, f, indent=2)

    # ---- Step 5: LLM predictions ----
    print(f"[{shots}-shot] Step 5/7: LLM predictions on selected nodes")
    llm_pred_jsonl = work_dir / f"{run_id}_selected_llm_preds.jsonl"
    generate_predictions(model=mdl, tokenizer=tokenizer,
                         input_json=selected_dataset_json, output_jsonl=llm_pred_jsonl, max_new_tokens=16)

    # ---- Step 6: WSFT/DPO data + training ----
    print(f"[{shots}-shot] Step 6/7: WSFT/DPO")
    dpo_json = data_dir / f"{run_id}_dpo.json"
    wsft_json = data_dir / f"{run_id}_wsft.json"

    wsft_cmd = [
        sys.executable, "create_wsft.py",
        "--dataset", DATASET,
        "--selected_nodes_path", str(ordered_json),
        "--llm_predictions", str(llm_pred_jsonl),
        "--dpo_output_path", str(dpo_json),
        "--sft_output_path", str(wsft_json),
        "--confidence_threshold", str(CONFIDENCE_THRESHOLD),
        "--shots", str(shots), "--gnn_type", "GCN",
        "--hidden_dim", "64", "--n_layers", "2",
        "--seed", str(SEED), "--device", DEVICE, "--path_prefix", ".",
    ]
    if JUDGE_MODE == "Hetero-Ensemble":
        wsft_cmd += ["--hetero_models", "GCN", "GAT", "SAGE",
                     "--ensemble_model_dir", str(BASE_DIR / "results" / "GNN"),
                     "--voting_method", VOTING_METHOD]
    elif JUDGE_MODE == "Homo-Ensemble":
        wsft_cmd += ["--pretrained_model", str(BASE_DIR / "results" / "GNN" / f"{DATASET}_{shots}_shot_best_model_ensemble0.pt"),
                     "--use_ensemble", "--ensemble_k", str(HOMO_ENSEMBLE_K),
                     "--ensemble_model_dir", str(BASE_DIR / "results" / "GNN"),
                     "--voting_method", VOTING_METHOD]
    else:
        wsft_cmd += ["--pretrained_model", str(BASE_DIR / "results" / "GNN" / f"{DATASET}_{shots}_shot_best_model_run0.pt")]

    subprocess.run(wsft_cmd, cwd=str(BASE_DIR), check=True)

    train_sft(model=mdl, tokenizer=tokenizer, json_path=wsft_json,
              output_dir=wsft_adapter_dir, epochs=WSFT_EPOCHS, lr=5e-5, grad_accum=4)

    if RUN_DPO_STAGE:
        dpo_rows = json.load(open(dpo_json, encoding="utf-8"))
        num_pref = sum(1 for r in dpo_rows if r.get("chosen", {}).get("value") != r.get("rejected", {}).get("value"))
        if num_pref > 0:
            run_dpo_update(model=mdl, tokenizer=tokenizer, dpo_json=dpo_json,
                           output_dir=dpo_adapter_dir, epochs=DPO_EPOCHS, beta=0.1, lr=1e-6, grad_accum=2)

    # ---- Step 7: Evaluation ----
    print(f"[{shots}-shot] Step 7/7: Evaluation")
    pred_jsonl = work_dir / "test_predictions.jsonl"
    generate_predictions(model=mdl, tokenizer=tokenizer, input_json=sft_test_json,
                         output_jsonl=pred_jsonl, max_samples=MAX_TEST_SAMPLES, max_new_tokens=16)

    eval_dir = work_dir / "evaluation"
    subprocess.run([
        sys.executable, "evaluate_predictions.py",
        "--dataset", DATASET, "--pred_file", str(pred_jsonl),
        "--output_dir", str(eval_dir),
        "--model_name", "result", "--device", DEVICE, "--path_prefix", ".",
    ], cwd=str(BASE_DIR), check=True)

    metrics = json.load(open(eval_dir / "result" / "metrics.json"))

    # Cleanup
    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


# ---- Run all shots ----
all_results = {}
for s in SHOTS_LIST:
    m = run_pipeline_for_shots(s)
    all_results[s] = m
    print(f"\n  [{s}-shot] Accuracy={m.get('accuracy', 0):.4f}  Macro-F1={m.get('macro_f1', 0):.4f}")

# ---- Summary table ----
print(f"\n{'='*60}")
print(f"  Multi-shot Summary | {JUDGE_MODE} | {DATASET} | seed={SEED}")
print(f"{'='*60}")
print(f"{'Shots':>8} {'Accuracy':>10} {'Macro-F1':>10}")
print(f"{'-'*30}")
for s in SHOTS_LIST:
    m = all_results[s]
    print(f"{s:>8} {m.get('accuracy', 0):>10.4f} {m.get('macro_f1', 0):>10.4f}")
print(f"{'='*60}")